In [98]:
import numpy as np
import featuregraph as fg

from featuregraph import training_cycle, test_cycle

In [99]:
task = {
    "train": [
        {
            "input": [[1, 2], [3, 4]],
            "output": [[1, 2, 2, 1], [3, 4, 4, 3]],
        }
    ],
    "test": [
        {
            "input": [[5, 6], [7, 8]],
        }
    ],
}

In [100]:
def get_training_pairs(task):
    return [
        {
            "grid": np.asarray(pair["input"], dtype=int),
            "output": np.asarray(pair["output"], dtype=int),
        }
        for pair in task["train"]
    ]

In [101]:
training_pairs = get_training_pairs(task)
instruction_layout = training_cycle(training_pairs)

In [102]:
training_pairs

[{'grid': array([[1, 2],
         [3, 4]]),
  'output': array([[1, 2, 2, 1],
         [3, 4, 4, 3]])}]

In [103]:
instruction_layout

array([['copy', 'flip_horizontal']], dtype=object)

In [104]:
def get_test_grids(task):
    return [
        np.asarray(pair["input"], dtype=int)
        for pair in task["test"]
    ]

In [105]:
test_grids = get_test_grids(task)
test_grids

[array([[5, 6],
        [7, 8]])]

In [106]:
predictions = [
    test_cycle(test_grid, instruction_layout)
    for test_grid in test_grids
]

predictions

[array([[5, 6, 6, 5],
        [7, 8, 8, 7]])]

In [107]:
def solve_task(task):
    training_pairs = get_training_pairs(task)
    instruction_layout = training_cycle(training_pairs)

    test_grids = get_test_grids(task)

    return [
        test_cycle(test_grid, instruction_layout)
        for test_grid in test_grids
    ]

In [108]:
task_predictions = solve_task(task)

assert len(task_predictions) == len(task["test"])
assert np.array_equal(task_predictions[0], predictions[0])

task_predictions

[array([[5, 6, 6, 5],
        [7, 8, 8, 7]])]

In [109]:
def predictions_to_lists(predictions):
    return [
        prediction.tolist()
        for prediction in predictions
    ]

In [110]:
serialized_predictions = predictions_to_lists(task_predictions)
serialized_predictions

[[[5, 6, 6, 5], [7, 8, 8, 7]]]

In [111]:
assert isinstance(serialized_predictions, list)
assert isinstance(serialized_predictions[0], list)
assert isinstance(serialized_predictions[0][0][0], int)

In [112]:
def predictions_to_attempts(predictions):
    return [
        {
            "attempt_1": prediction.tolist(),
            "attempt_2": prediction.tolist(),
        }
        for prediction in predictions
    ]

In [113]:
attempts = predictions_to_attempts(task_predictions)
attempts

[{'attempt_1': [[5, 6, 6, 5], [7, 8, 8, 7]],
  'attempt_2': [[5, 6, 6, 5], [7, 8, 8, 7]]}]

In [114]:
assert len(attempts) == len(task["test"])
assert attempts[0]["attempt_1"] == serialized_predictions[0]
assert attempts[0]["attempt_2"] == serialized_predictions[0]

In [115]:
task_id = "example1"

submission = {
    task_id: attempts
}

submission

{'example1': [{'attempt_1': [[5, 6, 6, 5], [7, 8, 8, 7]],
   'attempt_2': [[5, 6, 6, 5], [7, 8, 8, 7]]}]}

In [116]:
import json

submission_json = json.dumps(submission)

assert isinstance(submission_json, str)
assert json.loads(submission_json) == submission

submission_json

'{"example1": [{"attempt_1": [[5, 6, 6, 5], [7, 8, 8, 7]], "attempt_2": [[5, 6, 6, 5], [7, 8, 8, 7]]}]}'

In [117]:
challenges = {
    "example1": task,
    "example2": task,
}

In [118]:
def solve_challenges(challenges):
    submission = {}

    for task_id, task in challenges.items():
        predictions = solve_task(task)
        submission[task_id] = predictions_to_attempts(predictions)

    return submission

In [119]:
submission = solve_challenges(challenges)
submission

{'example1': [{'attempt_1': [[5, 6, 6, 5], [7, 8, 8, 7]],
   'attempt_2': [[5, 6, 6, 5], [7, 8, 8, 7]]}],
 'example2': [{'attempt_1': [[5, 6, 6, 5], [7, 8, 8, 7]],
   'attempt_2': [[5, 6, 6, 5], [7, 8, 8, 7]]}]}

In [120]:
assert set(submission) == set(challenges)

for task_id, task in challenges.items():
    assert len(submission[task_id]) == len(task["test"])
    assert set(submission[task_id][0]) == {
        "attempt_1",
        "attempt_2",
    }

In [121]:
def fallback_predictions(task):
    return [
        np.asarray(pair["input"], dtype=int)
        for pair in task["test"]
    ]

In [122]:
def solve_challenges(challenges):
    submission = {}
    failures = {}

    for task_id, task in challenges.items():
        try:
            predictions = solve_task(task)
        except ValueError as error:
            predictions = fallback_predictions(task)
            failures[task_id] = str(error)

        submission[task_id] = predictions_to_attempts(predictions)

    return submission, failures

In [123]:
submission, failures = solve_challenges(challenges)

assert failures == {}
assert set(submission) == set(challenges)

failures

{}

In [124]:
unsupported_task = {
    "train": [
        {
            "input": [[1, 2], [3, 4]],
            "output": [[9, 9], [9, 9]],
        }
    ],
    "test": [
        {
            "input": [[5, 6], [7, 8]],
        }
    ],
}

In [125]:
mixed_challenges = {
    "supported": task,
    "unsupported": unsupported_task,
}

submission, failures = solve_challenges(mixed_challenges)

In [126]:
failures

{'unsupported': 'No operator matched block (0, 0).'}

In [127]:
assert "supported" not in failures
assert "unsupported" in failures

assert submission["unsupported"][0]["attempt_1"] == [
    [5, 6],
    [7, 8],
]

assert set(submission) == set(mixed_challenges)

In [128]:
from pathlib import Path
import json


def write_submission(submission, path):
    path = Path(path)

    with path.open("w", encoding="utf-8") as file:
        json.dump(submission, file)

    return path

In [129]:
submission_path = write_submission(
    submission,
    "submission.json",
)

submission_path

PosixPath('submission.json')

In [130]:
with submission_path.open(encoding="utf-8") as file:
    loaded_submission = json.load(file)

assert loaded_submission == submission
assert set(loaded_submission) == set(mixed_challenges)

loaded_submission

{'supported': [{'attempt_1': [[5, 6, 6, 5], [7, 8, 8, 7]],
   'attempt_2': [[5, 6, 6, 5], [7, 8, 8, 7]]}],
 'unsupported': [{'attempt_1': [[5, 6], [7, 8]],
   'attempt_2': [[5, 6], [7, 8]]}]}

In [131]:
def load_challenges(path):
    path = Path(path)

    with path.open(encoding="utf-8") as file:
        return json.load(file)

In [132]:
fixture_path = Path("challenges_fixture.json")

with fixture_path.open("w", encoding="utf-8") as file:
    json.dump(mixed_challenges, file)

loaded_challenges = load_challenges(fixture_path)

assert loaded_challenges == mixed_challenges
assert set(loaded_challenges) == {
    "supported",
    "unsupported",
}

In [133]:
submission, failures = solve_challenges(loaded_challenges)
submission_path = write_submission(submission, "submission.json")

assert submission_path.exists()

failures

{'unsupported': 'No operator matched block (0, 0).'}

In [134]:
def run_harness(challenges_path, submission_path):
    challenges = load_challenges(challenges_path)
    submission, failures = solve_challenges(challenges)
    written_path = write_submission(submission, submission_path)

    return {
        "submission_path": written_path,
        "number_of_tasks": len(challenges),
        "number_solved": len(challenges) - len(failures),
        "number_failed": len(failures),
        "failures": failures,
    }

In [135]:
report = run_harness(
    "challenges_fixture.json",
    "submission.json",
)

report

{'submission_path': PosixPath('submission.json'),
 'number_of_tasks': 2,
 'number_solved': 1,
 'number_failed': 1,
 'failures': {'unsupported': 'No operator matched block (0, 0).'}}

In [136]:
assert report["number_of_tasks"] == 2
assert report["number_solved"] == 1
assert report["number_failed"] == 1
assert report["submission_path"].exists()

In [137]:
challenges_path = Path(
    "/kaggle/input/arc-prize-2026/arc-agi_test_challenges.json"
)

In [138]:
list(Path(".").rglob("*challenges*.json"))

[PosixPath('challenges_fixture.json')]

In [139]:
# report = run_harness(
#     challenges_path,
#     "submission.json",
# )

# report

In [140]:
challenge_files = list(Path(".").rglob("*challenges*.json"))
challenge_files

[PosixPath('challenges_fixture.json')]

In [141]:
# with Path("data/arc-agi-2/00576224.json").open(encoding="utf-8") as file:
#     real_task = json.load(file)

# real_challenges = {
#     "00576224": real_task,
# }

In [142]:
Path.cwd()

PosixPath('/workspaces/codespaces-jupyter/notebooks')

In [143]:
real_task_path = Path("../data/arc-agi-2/00576224.json")

assert real_task_path.exists()

with real_task_path.open(encoding="utf-8") as file:
    real_task = json.load(file)

real_challenges = {
    "00576224": real_task,
}

In [144]:
submission, failures = solve_challenges(real_challenges)

failures

{}

In [145]:
predicted_output = submission["00576224"][0]["attempt_1"]
expected_output = real_task["test"][0]["output"]

is_correct = predicted_output == expected_output
is_correct

True

In [146]:
print("Predicted:")
print(np.asarray(predicted_output))

print("\nExpected:")
print(np.asarray(expected_output))

Predicted:
[[3 2 3 2 3 2]
 [7 8 7 8 7 8]
 [2 3 2 3 2 3]
 [8 7 8 7 8 7]
 [3 2 3 2 3 2]
 [7 8 7 8 7 8]]

Expected:
[[3 2 3 2 3 2]
 [7 8 7 8 7 8]
 [2 3 2 3 2 3]
 [8 7 8 7 8 7]
 [3 2 3 2 3 2]
 [7 8 7 8 7 8]]


In [147]:
assert np.array_equal(
    np.asarray(predicted_output),
    np.asarray(expected_output),
)

In [148]:
def run_harness(challenges_path, submission_path):
    challenges = load_challenges(challenges_path)
    submission, failures = solve_challenges(challenges)
    written_path = write_submission(submission, submission_path)

    return {
        "submission_path": written_path,
        "number_of_tasks": len(challenges),
        "number_supported": len(challenges) - len(failures),
        "number_fallback": len(failures),
        "failures": failures,
    }

In [149]:
# assert report["number_of_tasks"] == 2
# assert report["number_supported"] == 1
# assert report["number_fallback"] == 1

In [150]:
report = run_harness(
    "challenges_fixture.json",
    "submission.json",
)

report

{'submission_path': PosixPath('submission.json'),
 'number_of_tasks': 2,
 'number_supported': 1,
 'number_fallback': 1,
 'failures': {'unsupported': 'No operator matched block (0, 0).'}}

In [151]:
assert report["number_of_tasks"] == 2
assert report["number_supported"] == 1
assert report["number_fallback"] == 1